In [104]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,roc_auc_score,roc_curve)


In [105]:
df = pd.read_csv("../kaggle_files/Titanic dataset zip/Titanic.csv")


In [106]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sex       891 non-null    str    
 1   age       714 non-null    float64
 2   sibsp     891 non-null    int64  
 3   parch     891 non-null    int64  
 4   fare      891 non-null    float64
 5   embarked  889 non-null    str    
 6   class     891 non-null    str    
 7   who       891 non-null    str    
 8   alone     891 non-null    bool   
 9   survived  891 non-null    int64  
dtypes: bool(1), float64(2), int64(3), str(4)
memory usage: 63.6 KB


In [107]:
df['age'] = df['age'].fillna(df['age'].mean())

In [108]:
X = df.drop(columns=['survived'])
Y= df['survived']


In [109]:
X_train, X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [110]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sex       891 non-null    str    
 1   age       891 non-null    float64
 2   sibsp     891 non-null    int64  
 3   parch     891 non-null    int64  
 4   fare      891 non-null    float64
 5   embarked  889 non-null    str    
 6   class     891 non-null    str    
 7   who       891 non-null    str    
 8   alone     891 non-null    bool   
 9   survived  891 non-null    int64  
dtypes: bool(1), float64(2), int64(3), str(4)
memory usage: 63.6 KB


In [111]:
X_train['age'].skew()

np.float64(0.3599658846236587)

In [112]:
X_train['parch'].skew()

np.float64(2.6954589914062086)

In [113]:
X_train['sibsp'].skew()

np.float64(3.619385065323857)

In [114]:
X_train['fare'].skew()

np.float64(4.875065571137606)

In [115]:
X_train['sex'].value_counts()

sex
male      467
female    245
Name: count, dtype: int64

In [116]:
X_train['embarked'].value_counts()

embarked
S    525
C    125
Q     60
Name: count, dtype: int64

In [117]:
X_train['class'].value_counts()

class
Third     398
First     163
Second    151
Name: count, dtype: int64

In [118]:
X_train['who'].value_counts()

who
man      432
woman    211
child     69
Name: count, dtype: int64

In [119]:
X_train['alone'].value_counts()

alone
True     429
False    283
Name: count, dtype: int64

In [120]:
df

,sex,age,sibsp,parch,fare,embarked,class,who,alone,survived
0,male,22.000000,1,0,7.2500,S,Third,man,False,0
1,female,38.000000,1,0,71.2833,C,First,woman,False,1
2,female,26.000000,0,0,7.9250,S,Third,woman,True,1
3,female,35.000000,1,0,53.1000,S,First,woman,False,1
4,male,35.000000,0,0,8.0500,S,Third,man,True,0
...,...,...,...,...,...,...,...,...,...,...
886,male,27.000000,0,0,13.0000,S,Second,man,True,0
887,female,19.000000,0,0,30.0000,S,First,woman,True,1
888,female,29.699118,1,2,23.4500,S,Third,woman,False,0
889,male,26.000000,0,0,30.0000,C,First,man,True,1


In [121]:
nominal_encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=True)
nominal_category_columns = ['sex','embarked','who','alone']
X_train_nominal_encoded = nominal_encoder.fit_transform(X_train[nominal_category_columns])
X_test_nominal_encoded =  nominal_encoder.transform(X_test[nominal_category_columns])

In [122]:
ordinal_encoder  = OrdinalEncoder( categories=[['Third','Second','First']])
ordinal_category_columns = ['class']
X_train_ordinal_encoded = ordinal_encoder.fit_transform(X_train[ordinal_category_columns])
X_test_ordinal_encoded = ordinal_encoder.transform(X_test[ordinal_category_columns])

In [123]:
numerical_Transform = PowerTransformer(method='yeo-johnson')
numerical_category_columns = ['age','sibsp','parch','fare']

X_train_numerical_Transformed = numerical_Transform.fit_transform(X_train[numerical_category_columns])
X_test_numerical_Transformed = numerical_Transform.transform(X_test[numerical_category_columns])


In [124]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_numerical_Transformed)
X_test_scaled = scaler.transform(X_test_numerical_Transformed)

In [125]:
X_train_nominal_df = pd.DataFrame.sparse.from_spmatrix(X_train_nominal_encoded, columns= nominal_encoder.get_feature_names_out(nominal_category_columns),index=X_train.index)
X_train_ordinal_df = pd.DataFrame(X_train_ordinal_encoded,columns=ordinal_category_columns,index=X_train.index)


In [126]:
X_train_numerical_df = pd.DataFrame(X_train_scaled,columns=numerical_category_columns,index=X_train.index)


In [127]:
X_test_nominal_df = pd.DataFrame.sparse.from_spmatrix(X_test_nominal_encoded, columns=  
                    nominal_encoder.get_feature_names_out(nominal_category_columns),index=X_test.index)
X_test_ordinal_df = pd.DataFrame(X_test_ordinal_encoded,columns=ordinal_category_columns,index=X_test.index)
X_test_numerical_df = pd.DataFrame(X_test_scaled,columns=numerical_category_columns,index=X_test.index)

In [128]:
X_train_final = pd.concat([X_train_numerical_df, X_train_nominal_df, X_train_ordinal_df],axis=1)


In [129]:
X_test_final = pd.concat([X_test_numerical_df, X_test_nominal_df, X_test_ordinal_df ],axis=1)

In [130]:
X_train_final.head(5)

,age,sibsp,parch,fare,sex_female,sex_male,embarked_C,embarked_Q,embarked_S,embarked_nan,who_child,who_man,who_woman,alone_False,alone_True,class
331,1.206781,-0.683427,-0.561965,0.479998,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,2.0
733,-0.469258,-0.683427,-0.561965,-0.283753,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,1.0
382,0.221294,-0.683427,-0.561965,-0.772447,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,0.0
704,-0.235345,1.350891,-0.561965,-0.781285,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,0.0
813,-1.911311,1.750636,1.841200,0.568460,1.0,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN,0.0


In [131]:
model = LogisticRegression(class_weight="balanced")

In [132]:
model.fit(X_train_final,Y_train)

c:\Users\Princ\OneDrive\Documents\Projects\python\DataScience\.tutorial_class\Lib\site-packages\sklearn\utils\validation.py:911: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values